In [ ]:
pana_dataset_path = "../hf-dataset/panasonic_qa_v1_test"
model_path = "../gallery/Qwen3-4B-Instruct-2507-ft-panasonic_qa_v1_train-2x32-sft"

In [2]:
from datasets import load_from_disk

ds = load_from_disk(pana_dataset_path)
ds

/home/parsa/.conda/envs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['question', 'answer', 'documents'],
    num_rows: 1000
})

In [5]:
from utils import start_vllm_service, start_vllm_serviceV2 ,start_vllm_servicev3
# (proc, base_url) = start_vllm_servicev3(model_path, lora_path,port=8182)
(proc, base_url) = start_vllm_service(model_path,port=8182)

Waiting for vllm to launch. Retrying in 10 seconds


(APIServer pid=1178624) The tokenizer you are loading from '../gallery/Qwen3-4B-Instruct-2507-ft-panasonic_qa_v1_train-2x32-sft' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Waiting for vllm to launch. Retrying in 10 seconds
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.74s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.39s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.44s/it]
(EngineCore_DP0 pid=1178810) 


Waiting for vllm to launch. Retrying in 10 seconds


(EngineCore_DP0 pid=1178810) 2026-01-13 10:45:22,908 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=1178810) 2026-01-13 10:45:22,917 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 42.39it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 51.50it/s]
(EngineCore_DP0 pid=1178810) The tokenizer you are loading from '../gallery/Qwen3-4B-Instruct-2507-ft-panasonic_qa_v1_train-2x32-sft' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
(APIServer pid=1178624) INFO:     Started server process [1178624]
(APIServer pid=1178624) INFO:     Waiting for applica

(APIServer pid=1178624) INFO:     127.0.0.1:47212 - "GET /v1/models HTTP/1.1" 200 OK
Model ../gallery/Qwen3-4B-Instruct-2507-ft-panasonic_qa_v1_train-2x32-sft is on http://127.0.0.1:8182/v1/


In [5]:
from openai import OpenAI
# Inference
client = OpenAI(
    base_url=base_url,
    api_key="none"
)

# Model sees as a chat completion
def chat_completion(prompt, max_tokens=1024, temperature=0.0):

    msg = [
        {"role": "system", "content": 'You are a helpful assistant. You must output your answer strictly as valid JSON in the format {"answer": ["choice"]}.'},
        {"role": "user", "content": prompt},
    ]

    response = client.chat.completions.create(
        model="test",
        messages=msg,
        max_tokens=max_tokens,
        temperature=temperature,
        # stop=["\n\n"]  # Optional: stop sequences
    )
    return response.choices[0].message.content

In [6]:
def normalize_documents(documents):
    flat_docs = []
    for doc in documents:
        if isinstance(doc, list):
            flat_docs.append(" ".join(map(str, doc)))
        elif isinstance(doc, dict):
            flat_docs.append(" ".join(f"{k}: {v}" for k, v in doc.items()))
        else:
            flat_docs.append(str(doc))
    return flat_docs

def formatting_prompts_func(example):
    question = example["question"]
    documents = example["documents"]
    answer = str(example["answer"])
    prompt = """
        Based on relevat document answer this question.
        relevant document: {}
        question: {}
    """
    input = prompt.format("\n".join(documents), question)

    return {"prompt" : input, "answer": answer}

ds = ds.map(formatting_prompts_func)

In [7]:
import concurrent.futures
from tqdm import tqdm

# 1. Define a helper function to process a single sample
def process_sample(sample):
    q = sample["question"]
    a = sample["answer"]
    # This is where the time-consuming API call happens
    p = chat_completion(q)
    # Return both so we keep them linked
    return p, a

pred = []
refs = []

# 2. Configure the number of parallel workers
# Adjust max_workers based on your API rate limits (e.g., 5, 10, or 20)
MAX_WORKERS = 32 

print(f"Starting evaluation with {MAX_WORKERS} threads...")

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 3. Submit tasks and map over the dataset
    # executor.map preserves the original order of the dataset
    results = list(tqdm(executor.map(process_sample, ds), total=len(ds), desc="Evaluating"))

# 4. Unpack results
for p, a in results:
    pred.append(p)
    refs.append(a)

print("Evaluation complete.")

Starting evaluation with 32 threads...


Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating:   1%|          | 6/1000 [00:00<01:20, 12.42it/s]

(APIServer pid=1041103) INFO:     127.0.0.1:40278 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40270 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40292 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40314 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40358 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40372 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40398 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40404 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40418 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40424 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.

Evaluating:   4%|▎         | 35/1000 [00:00<00:13, 72.19it/s]

...

(APIServer pid=1041103) INFO:     127.0.0.1:40278 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40538 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40482 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40398 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40466 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40518 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40534 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40554 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40308 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40424 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.

Evaluating:  85%|████████▌ | 851/1000 [00:06<00:00, 183.09it/s]

(APIServer pid=1041103) INFO:     127.0.0.1:40292 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40314 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=1041103) INFO:     127.0.0.1:40538 - "POST /v1/chat/completions HTTP/1.1" 200 OK


Evaluating: 100%|██████████| 1000/1000 [00:06<00:00, 152.28it/s]

(APIServer pid=1041103) INFO:     127.0.0.1:40278 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Evaluation complete.


In [8]:
import re, json
json_pred = []
for item in pred:
    try:
        json_blocks = re.findall(r'\{.*?\}', item, flags=re.DOTALL)
        if len(json_blocks) > 0:
            answer = json.loads(json_blocks[0])
            json_pred.append(str(answer["answer"]))
        else:
            json_pred.append(str(['none']))
    except Exception as e:
        print(json_blocks[0])
        json_pred.append(str(['none']))

len(json_pred)

{"answer": ["B"}
{"answer": ["B"}


1000

In [9]:
import ast

def compute_advanced_metrics(predictions, references):
    """
    Computes Exact Set-Match, Jaccard Similarity, and F1 Score.
    Returns metrics and detailed error lists for debugging.
    """
    set_match_scores = []
    jaccard_scores = []
    f1_scores = []
    
    # Store indices and details for debugging
    parsing_errors = []   # Format: {'index': i, 'pred': str, 'error': msg}
    incorrect_cases = []  # Format: {'index': i, 'pred': set, 'ref': set}

    for i, (pred_str, ref_str) in enumerate(zip(predictions, references)):
        try:
            # Parse strings to sets
            pred_set = set(ast.literal_eval(pred_str))
            ref_set = set(ast.literal_eval(ref_str))
            
            # --- Calculation Logic ---
            
            # 1. Exact Set Match
            is_match = pred_set == ref_set
            set_match_scores.append(1 if is_match else 0)
            
            if not is_match:
                incorrect_cases.append({
                    'index': i,
                    'prediction': pred_set,
                    'reference': ref_set
                })

            # 2. Jaccard Similarity
            intersection = len(pred_set.intersection(ref_set))
            union = len(pred_set.union(ref_set))
            jaccard = intersection / union if union > 0 else 0
            jaccard_scores.append(jaccard)

            # 3. F1 Score
            precision = intersection / len(pred_set) if len(pred_set) > 0 else 0
            recall = intersection / len(ref_set) if len(ref_set) > 0 else 0
            
            if (precision + recall) > 0:
                f1 = 2 * (precision * recall) / (precision + recall)
            else:
                f1 = 0
            f1_scores.append(f1)

        except Exception as e:
            # Handle parsing errors (score as 0)
            set_match_scores.append(0)
            jaccard_scores.append(0)
            f1_scores.append(0)
            
            # Record the parsing error index
            parsing_errors.append({
                'index': i,
                'prediction': pred_str,
                'error': str(e)
            })
            continue

    # Average over all samples
    metrics = {
        "set_match": sum(set_match_scores) / len(set_match_scores) if set_match_scores else 0,
        "jaccard": sum(jaccard_scores) / len(jaccard_scores) if jaccard_scores else 0,
        "f1": sum(f1_scores) / len(f1_scores) if f1_scores else 0
    }
    
    return metrics, parsing_errors, incorrect_cases

In [ ]:
metrics, parse_errs, wrong_ans = compute_advanced_metrics(json_pred, refs)

print("--- Metrics ---")
print(f"Set-Match Accuracy: {metrics['set_match']*100:.2f}%")
print(f"Jaccard Similarity: {metrics['jaccard']*100:.2f}%")
print(f"F1 Score:           {metrics['f1']*100:.2f}%")

output_data = {
    "Set-Match Accuracy": f"{metrics['set_match'] * 100:.2f}%",
    "Jaccard Similarity": f"{metrics['jaccard'] * 100:.2f}%",
    "F1 Score": f"{metrics['f1'] * 100:.2f}%"
}

file_name = f"{(lora_path.split("/"))[-1]}.json"
# Write to a JSON file
with open(file_name, "w") as json_file:
    json.dump(output_data, json_file, indent=4)


--- Metrics ---
Set-Match Accuracy: 28.20%
Jaccard Similarity: 41.27%
F1 Score:           46.27%


In [6]:
import os
import signal
os.killpg(os.getpgid(proc.pid), signal.SIGTERM)

[rank0]:[W113 10:45:41.253967341 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
(APIServer pid=1178624) INFO:     Shutting down
(APIServer pid=1178624) INFO:     Waiting for application shutdown.
(APIServer pid=1178624) INFO:     Application shutdown complete.


Here is a breakdown of the concept tailored for your paper, comparing your new approach against the standard metrics.
Metric Comparison for Multi-Answer Evaluation

In the context of Multiple-Choice QA where questions may have multiple correct answers (e.g., R={A,B}), standard metrics often fail to capture logical correctness due to strict formatting constraints. We compare three evaluation strategies: Exact Match, Classic Accuracy, and the proposed Set-Match Accuracy.
1. Exact Match (EM)

    Definition: A binary metric measuring whether the predicted string sequence is character-for-character identical to the reference string.

    Formal Definition: EM(y,y^​)=1 if ystring​≡y^​string​ else 0

    Limitation: It is brittle to permutation and formatting. If the ground truth is ['A', 'B'] and the model predicts ['B', 'A'], EM assigns a score of 0, despite the answer being logically correct.

2. Classic Accuracy

    Definition: Typically used for single-label classification, this metric computes the fraction of instances where the predicted class matches the reference.

    Behavior in this Context: When applied to string representations of lists, Classic Accuracy behaves identically to Exact Match. It treats the entire string ['A', 'B'] as a single, indivisible class label.

    Limitation: It fails to account for the set-theoretic nature of the task, penalizing correct answers simply for differing element order.

3. Set-Match Accuracy (Proposed)

    Definition: A domain-specific metric that parses string representations into unordered sets before comparison. It verifies set equality, making the evaluation invariant to element order and minor formatting artifacts (e.g., whitespace).

    Formal Definition:
    Accset​=N1​i=1∑N​I(set(y^​i​)=set(yi​))

    Where I is the indicator function, y^​ is the predicted list, and y is the reference list.

    Advantage: This aligns the evaluation with the logic of the task. It ensures the model is assessed on its ability to identify the correct options, rather than its adherence to a specific sorting order (e.g., treating ['B', 'A'] as equivalent to ['A', 'B']).

Summary Table
Metric	Input Type	Correctness Condition	Handling of Permutation (['A','B'] vs ['B','A'])
Exact Match	String	String(y^​)≡String(y)	Fail (0)
Classic Accuracy	String/Label	Label(y^​)≡Label(y)	Fail (0)
Set-Match	Set	Set(y^​)=Set(y)	Pass (1)